In [16]:
import torch
import sys, os
import math as mt

src_dir = os.path.join(os.path.abspath(''), '..')  # goes up from models/ to src/
configs_dir = os.path.join(os.path.abspath(''), '../..')
sys.path.insert(0, os.path.abspath(src_dir))
sys.path.insert(0, os.path.abspath(configs_dir))

In [17]:
from models.autoencoder import Autoencoder
from models.dqn import DQN

K = 5
lr = 5e-4
train_epochs = 500
in_channels = 4 
num_channels = 10
kernel_dim = 2
maze_dim = 4
batch_size = 10
gamma_sparse = 1/20*mt.sqrt(maze_dim**2*in_channels/K)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [18]:
import numpy as np
import random
seed = 39
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

In [20]:
from torch.utils.data import TensorDataset
from configs.experiment_config import ExperimentConfig
from rl.teacher import value_iteration_all_tasks
from utils.generate_inventory_maps import load_maps
import importlib
import models.autoencoder as autoencoder_module
importlib.reload(autoencoder_module)
from models.autoencoder import trainSAE_inventory, testSAE_inventory


config = ExperimentConfig()

train_label_dict, train_wall_state_dict = load_maps(dataset_id='v1', split='train')

q_matrices, sopt_dict = value_iteration_all_tasks(train_label_dict, train_wall_state_dict, config, device)

sorted_tasks = sorted(q_matrices.keys())
q_tensors = torch.stack([q_matrices[t] for t in sorted_tasks])
labels_list = [train_label_dict[t] for t in sorted_tasks]
labels_tensor = torch.tensor(labels_list)
dataset = TensorDataset(q_tensors, labels_tensor[:,0], labels_tensor[:,1], labels_tensor[:,2], labels_tensor[:,3])


autoencoder = Autoencoder(in_channels, maze_dim, num_channels, kernel_dim, K)
optimizer = torch.optim.Adam(autoencoder.parameters(), lr=lr)

# training language
total_losses, recon_losses, sparsity_losses, all_messages = trainSAE_inventory(
    autoencoder, dataset, gamma_sparse, optimizer, batch_size, train_epochs, device
)

In [21]:
# getting messages from trained language
_, _, _, all_messages, message_dict = testSAE_inventory(
    autoencoder, dataset, batch_size, gamma_sparse, device
)

In [ ]:
import importlib
import evaluation.student_eval as student_eval
import src.env.inventory as inventory_module
importlib.reload(inventory_module)
from src.env.inventory import InventoryManagement
importlib.reload(student_eval)
from evaluation.student_eval import train_student_with_feedback_inventory, run_evaluations_inventory

student = DQN(K=config.K, n_actions=config.n_actions, device=device, input_dim=1)
autoencoder, student = student_eval.train_student_with_feedback_inventory(
    train_label_dict, 
    train_wall_state_dict, 
    sopt_dict, 
    q_matrices,
    autoencoder, 
    student, 
    config, 
    device,
)

Joint Training (Epochs):   0%|          | 0/750 [00:00<?, ?it/s]


RuntimeError: mat1 and mat2 shapes cannot be multiplied (16x6 and 7x10)

In [ ]:
# Extract the optimized messages generated by the newly trained Autoencoder
message_dict_feedback = {}
autoencoder.eval()
with torch.no_grad():
    for task, (wall_label, init_state, goal_state, demand) in train_label_dict.items():
        q_mat = q_matrices[task].unsqueeze(0).to(device)
        message, _ = autoencoder(q_mat)
        message_dict_feedback[(wall_label, init_state, goal_state, demand)] = message.squeeze(0).detach()

In [ ]:
informed_rates, misinformed_rates = student_eval.run_evaluations_inventory(
    student, train_label_dict, train_wall_state_dict, message_dict_feedback, sopt_dict, config, device
)

In [ ]:
from sklearn.decomposition import PCA
from utils.plotting import plot_pca_variance, plot_pca_by_label

# Stack all messages into a single tensor
messages = torch.cat(all_messages, dim=0)  # shape: (num_tasks, K)
messages_np = messages.numpy()

n_tasks = len(train_label_dict)
wall_labels   = [train_label_dict[i][0] for i in range(n_tasks)]
goal_labels   = [train_label_dict[i][2] for i in range(n_tasks)]
demand_labels = [train_label_dict[i][3] for i in range(n_tasks)]

pca = PCA(n_components=5)
pca_result = pca.fit_transform(messages_np)

# get plots
plot_pca_variance(pca)
plot_pca_by_label(pca_result, wall_labels,   "color: wall position")
plot_pca_by_label(pca_result, goal_labels,   "color: goal labels")
plot_pca_by_label(pca_result, demand_labels, "color: demand")